# Why did so many regular payments stop in August 2026?

**PNC Treasury Management · DSI · Payment Knowledge Graph**

`pkg_payment_regularity.ipynb` (§6d) found about twice the usual number of stopped regular **incoming** streams with an
expected payment in 2026-08, almost all of them **monthly** streams, spread over many payers (few payer-wide stops). This
notebook tests every explanation against the data, over the whole history (2024-01 onward), and ends with a verdict.

| Test | Question | What would prove it |
|---|---|---|
| §3 Seasonal | Does the same calendar month spike in other years? | August 2025 (and 2024) shows the same jump |
| §4 One source | Do the extra stops sit in one category, payer type or id type? Do pairs vanish from the edges for one category? | most of the excess in one place; one category losing pairs |
| §5 Replaced | Does the customer get the same payment from a new payer id within weeks? | many replacements: same name = id change, other name = real switch (new payroll provider, new tenant account) |
| §6 Real life | Do customers stay active? When in the month was the last payment? Which payers? | rent-like (1st–5th), lease or school calendars, named employers or agencies |

Streams are built exactly as in the regularity notebook (same definitions and settings), for incoming flows only (`in`
and `book`), for a `SAMPLE_PCT` sample of receiving customers with all their incoming streams. A stop is a regular
stream silent for `SKIP_MULT` × its median gap at the window end, dated by its expected payment: the same day one, two
or three months after the last payment for monthly, bimonthly and quarterly streams (they are paid on calendar dates),
last payment + median gap otherwise. Read-only: scratch files and PNG figures (`FIG_DIR`), no tables.

## Settings

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
DB = "bdahd01p_dlcdi1_cdi_tm"
SHUFFLE_PARTITIONS = 2000
HISTORY_START = "2024-01"     # first month of the edge tables
WINDOW_END    = None          # None = latest month published by pkg_edges_refresh
SAMPLE_PCT    = 10            # % of receiving customers (hash of mdm_id), with all their incoming streams

# stream definitions (same as pkg_payment_regularity.ipynb)
TOL_FRAC, TOL_MIN = 0.15, 1
SKIP_MULT     = 1.75
DAILY_STOP_BD = 5
MIN_PAYMENTS  = 4
REG_MIN       = 0.70
CADENCES = [("weekly", 6, 8), ("biweekly / semimonthly", 12, 17), ("monthly", 26, 35), ("bimonthly", 55, 66),
            ("quarterly", 84, 98)]

# the test
FOCUS         = "monthly"     # cadence whose spike is examined
SPIKE_MONTH   = "2026-08"     # month examined; None = month with the highest FOCUS stop rate (first 6 and last 2 left out)
REPL_BEFORE, REPL_AFTER = 45, 60   # §5: a replacement starts this many days before / after the expected payment
REPL_AMT_TOL  = 0.15          # §5: and its usual amount is within ±15% of the stopped stream's
MASS_STOP_MIN = 10            # payer-wide: the payer stopped ≥ this many sampled customers in the month

FIG_DIR      = "figures/pkg_stop_spike"
KEEP_SCRATCH = True

## Setup

In [ ]:
# ---- Setup ---------------------------------------------------------------------------------------
# Compatibility shims: Cloudera SPARK3 PySpark is older than the NumPy / pandas in the conda env.
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cat in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_cat)

import calendar, hashlib, json, time, datetime as dt, html as _html
from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder.appName("pkg_stop_spike_check")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "8g").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.executor.memoryOverhead", "3g")
         .config("spark.executor.cores", "2")
         .config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.ansi.enabled", "false")      # ratios with a zero denominator are NULL, not an error
spark.sparkContext.setLogLevel("ERROR")
# The cluster's Atlas lineage hook can't reach its Kafka topic and prints the full query plan on every write; netty /
# TransportResponseHandler print harmless errors when executors are released. Their loggers are switched off.
_QUIET = ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
          "org.apache.spark.sql.catalyst.util.package", "org.apache.spark.network", "io.netty")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in _QUIET:
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    try:
        _L1 = spark._jvm.org.apache.log4j
        for _n in _QUIET:
            _L1.LogManager.getLogger(_n).setLevel(_L1.Level.OFF)
    except Exception as _e:
        print("could not silence the Atlas logs:", str(_e)[:120])
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)

# ---- one consolidated HTML output per cell
def tbl(pdf, pct=(), dec=()):
    pdf = pdf.copy()
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.1%}"
        elif c in dec:                                 fmt[c] = "{:,.2f}"
        elif pd.api.types.is_bool_dtype(pdf[c]):       continue
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.0f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:
        sty = sty.hide(axis="index")
    except AttributeError:
        sty = sty.hide_index()
    sty = sty.set_table_styles([{"selector": "th, td", "props": "padding: 2px 8px; font-size: 12px;"}])
    return sty.to_html()
def sec(title, body):
    return f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>{body}"
def para(text):
    return f"<p style='margin:4px 0; font-size:13px'>{_html.escape(text)}</p>"
def out(*parts):
    display(HTML("<div>" + "".join(p for p in parts if p) + "</div>"))

# ---- HDFS (Hadoop FileSystem API through the JVM)
HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def delete(p):
    return fs_for(p).delete(JPath(p), True) if exists(p) else False
def size_gb(p):
    return fs_for(p).getContentSummary(JPath(p)).getLength() / 1e9 if exists(p) else 0.0
def ls_dirs(p):
    if not exists(p):
        return []
    return [s.getPath().getName() for s in fs_for(p).listStatus(JPath(p)) if s.isDirectory()]

TIMES = {}
class timed:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t0 = time.time()
    def __exit__(self, *a): TIMES[self.name] = (time.time() - self.t0) / 60

def month_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"
def month_range(a, b):
    out_, m = [], a
    while m <= b:
        out_.append(m); m = month_add(m, 1)
    return out_

def us_bank_holidays(y0, y1):
    """US Federal Reserve holidays (ACH / Fedwire closed). A holiday on a Sunday is observed on Monday; one on a
    Saturday is not observed (the Fed stays open on Friday)."""
    def nth(y, m, wd, k):                      # k-th weekday wd (Mon = 0) of the month; k = -1 → last
        if k > 0:
            d = dt.date(y, m, 1)
            d += dt.timedelta(days=(wd - d.weekday()) % 7 + 7 * (k - 1))
        else:
            d = dt.date(y, m, calendar.monthrange(y, m)[1])
            d -= dt.timedelta(days=(d.weekday() - wd) % 7)
        return d
    out_ = []
    for y in range(y0, y1 + 1):
        fixed = [dt.date(y, 1, 1), dt.date(y, 6, 19), dt.date(y, 7, 4), dt.date(y, 11, 11), dt.date(y, 12, 25)]
        moving = [nth(y, 1, 0, 3), nth(y, 2, 0, 3), nth(y, 5, 0, -1), nth(y, 9, 0, 1), nth(y, 10, 0, 2), nth(y, 11, 3, 4)]
        for d in fixed:
            if d.weekday() == 6:
                d += dt.timedelta(days=1)
            if d.weekday() < 5:
                out_.append(d)
        out_ += moving
    return sorted(out_)
HOLIDAYS = us_bank_holidays(2023, 2028)

def bidx(c):
    """Business-day index of a date column: each business day (Monday–Friday, not a Fed holiday) counts 1; weekends
    and holidays share the next business day's index. So Friday → Monday is 1 business day, Friday → Tuesday after a
    Monday holiday is also 1, and a payment made on a weekend or holiday is on time for a daily stream."""
    n = F.datediff(c, F.to_date(F.lit("1970-01-05")))           # 1970-01-05 is a Monday
    wk = F.floor(n / 7)
    hol = sum(F.when(c > F.to_date(F.lit(h.isoformat())), 1).otherwise(0) for h in HOLIDAYS)
    return (wk * 5 + F.least(n - wk * 7, F.lit(5)) - hol).cast("long")

print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__}")

## §1 Incoming stream days

Daily edges into sampled customers (own accounts left out), external payers at `ent_l2`, one row per stream and payment
day with the category, id source and rail of its largest payment, and the gap to the previous payment day.

In [ ]:
# ---- §1 Incoming stream days ----------------------------------------------------------------------
T_EDGE_D, T_EDGE_M = f"{DB}.pkg_edge_daily", f"{DB}.pkg_edge_monthly"
T_MAP, T_CUST, T_ENT = f"{DB}.pkg_cpty_entity_map_v2", f"{DB}.pkg_cust_dim", f"{DB}.pkg_cpty_entity_dim"
DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
def table_location(t):
    return [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Location"][0].rstrip("/")
def cols_of(t):
    try:
        return set(spark.table(t).columns)
    except Exception:
        return None

PROBLEMS = []
NEED = {T_EDGE_D: ["txn_date", "src_id", "src_kind", "dst_id", "dst_kind", "category", "cpty_id_src", "is_self", "amount",
                   "n_txn", "month"],
        T_EDGE_M: ["src_id", "src_kind", "dst_id", "dst_kind", "category", "is_self", "month"],
        T_MAP: ["cpty_id", "ent_l2", "name_class"], T_CUST: ["mdm_id", "party_type", "is_tm", "customer_name"],
        T_ENT: ["ent_id", "ent_name"]}
for t, need in NEED.items():
    have = cols_of(t)
    if have is None:
        PROBLEMS.append({"problem": "table missing", "detail": t}); continue
    PROBLEMS += [{"problem": "column missing", "detail": f"{t}.{c}"} for c in need if c not in have]
PUB = {}
for t in (T_EDGE_D, T_EDGE_M):
    loc = table_location(t)
    parts = sorted(r[0].split("=", 1)[1] for r in spark.sql(f"SHOW PARTITIONS {t}").collect())
    PUB[t] = [m for m in parts if exists(f"{loc}/month={m}/_PKG_BUILD")]
END = WINDOW_END or (PUB[T_EDGE_D][-1] if PUB[T_EDGE_D] else None)
MONTHS = month_range(HISTORY_START, END) if END else []
for t in (T_EDGE_D, T_EDGE_M):
    miss = [m for m in MONTHS if m not in PUB[t]]
    if miss:
        PROBLEMS.append({"problem": f"months not published in {t}", "detail": ", ".join(miss)})
if PROBLEMS:
    out(sec(f"§1 stopped: {len(PROBLEMS)} problem(s)", tbl(pd.DataFrame(PROBLEMS))))
    raise AssertionError(f"§1: {len(PROBLEMS)} problem(s), see the table above")

W_END = dt.date(int(END[:4]), int(END[5:7]), calendar.monthrange(int(END[:4]), int(END[5:7]))[1])
VERSION = "spike_v1"
SIG = hashlib.md5(json.dumps({"v": VERSION, "m": MONTHS, "s": SAMPLE_PCT, "h": [h.isoformat() for h in HOLIDAYS]})
                  .encode()).hexdigest()[:10]
SCR = f"{DB_LOC}/_tmp_pkg_stop_spike"
P_SD, P_ST, P_PAIRS = f"{SCR}/sd_{SIG}", f"{SCR}/st_{SIG}", f"{SCR}/pairs_{SIG}"
K = ["payer", "payer_kind", "payee", "flow"]
def sampled(c):
    return F.lit(True) if SAMPLE_PCT >= 100 else F.expr(f"pmod(xxhash64({c}), 100) < {int(SAMPLE_PCT)}")
code_of = F.split(F.col("category"), r"\.").getItem(0)

sd_built = not exists(f"{P_SD}/_SUCCESS")
if sd_built:
    with timed("§1 stream days"):
        e = (spark.table(T_EDGE_D).where(F.col("month").isin(MONTHS))
             .where((F.col("dst_kind") == "CUST") & ~F.coalesce(F.col("is_self"), F.lit(False)) & sampled("dst_id"))
             .withColumn("flow", F.when(F.col("src_kind") == "CUST", "book").otherwise("in"))
             .select("txn_date", "src_id", "src_kind", "dst_id", "flow", code_of.alias("cat"),
                     F.coalesce(F.col("cpty_id_src"), F.lit("internal")).alias("idsrc"),
                     F.col("amount").cast("double").alias("amount"), F.col("n_txn").cast("long").alias("n_txn")))
        mp = spark.table(T_MAP).select(F.col("cpty_id").alias("src_id"), F.col("ent_l2").alias("cp_ent"),
                                       F.col("name_class").alias("cp_class"))
        e = (e.where(F.col("src_kind") == "CPTY").join(mp, "src_id", "left")
              .unionByName(e.where(F.col("src_kind") == "CUST").withColumn("cp_ent", F.lit(None).cast("string"))
                            .withColumn("cp_class", F.lit(None).cast("string"))))
        e = (e.withColumn("payer", F.when(F.col("src_kind") == "CPTY", F.coalesce("cp_ent", "src_id")).otherwise(F.col("src_id")))
              .withColumnRenamed("src_kind", "payer_kind").withColumnRenamed("dst_id", "payee"))
        sd = e.groupBy(*K, "txn_date").agg(
            F.sum("amount").alias("amt"), F.sum("n_txn").alias("n_txn"),
            F.max(F.struct("amount", "cat", "idsrc", "cp_class")).alias("_m"),
            F.countDistinct("src_id").alias("n_ids"))
        sd = sd.select(*K, "txn_date", "amt", "n_txn", "n_ids", F.col("_m.cat").alias("cat"), F.col("_m.idsrc").alias("idsrc"),
                       F.col("_m.cp_class").alias("cp_class"))
        w = Window.partitionBy("payer", "payee").orderBy("txn_date")
        sd = (sd.withColumn("prev_dt", F.lag("txn_date").over(w))
                .withColumn("gap", F.datediff("txn_date", "prev_dt"))
                .withColumn("gap_bd", bidx(F.col("txn_date")) - bidx(F.col("prev_dt"))))
        sd.write.mode("overwrite").parquet(P_SD)
SD = spark.read.parquet(P_SD)
s1 = (SD.groupBy("flow").agg(F.count("*").alias("payment_days"), (F.sum("amt") / 1e9).alias("dollars_B"),
                             F.countDistinct("payee").alias("customers")).toPandas())
_S1_OK = True
out(sec(f"§1 Incoming stream days {MONTHS[0]} → {MONTHS[-1]}, {SAMPLE_PCT}% of receiving customers — "
        + (f"built ({TIMES['§1 stream days']:.1f} min)" if sd_built else "reused"), tbl(s1, dec=("dollars_B",))))

## §2 Streams

One row per incoming stream over the whole history: cadence, regularity, stop and expected payment, the category, id
source and payer type of its last payment, and the receiving customer's segment. Payer names come from
`pkg_cpty_entity_dim` (external) or `pkg_cust_dim` (book).

In [ ]:
# ---- §2 Streams ---------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

st_built = not exists(f"{P_ST}/_SUCCESS")
if st_built:
    with timed("§2 streams"):
        sd = spark.read.parquet(P_SD)
        s1 = sd.groupBy(*K).agg(
            F.count("*").alias("n_pay"), F.sum("amt").alias("amt"), F.min("txn_date").alias("first_dt"),
            F.max("txn_date").alias("last_dt"), F.percentile_approx("gap", 0.5, 1000).alias("med_gap"),
            F.percentile_approx("gap_bd", 0.5, 1000).alias("med_gap_bd"), F.percentile_approx("amt", 0.5, 1000).alias("med_amt"),
            F.max(F.struct("txn_date", "cat", "idsrc", "cp_class", "amt")).alias("_last"), F.max("n_ids").alias("max_ids"))
        j = sd.join(s1.select(*K, "med_gap", "med_gap_bd"), K)
        daily = F.col("med_gap_bd") <= 1
        tol = F.greatest(F.lit(float(TOL_MIN)), F.col("med_gap") * float(TOL_FRAC))
        ont = F.when(daily, F.col("gap_bd") <= 1).otherwise(F.abs(F.col("gap") - F.col("med_gap")) <= tol)
        a = j.groupBy(*K).agg(F.sum(F.col("gap").isNotNull().cast("long")).alias("n_gaps"),
                              F.sum(F.coalesce(F.col("gap").isNotNull() & ont, F.lit(False)).cast("long")).alias("n_ontime"))
        cu = (spark.table(T_CUST).select(F.col("mdm_id").alias("payee"), F.upper(F.trim("party_type")).alias("pt"),
                                         F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("tm"))
                   .dropDuplicates(["payee"]))
        st = (s1.join(a, K).join(cu, "payee", "left")
                .select(*K, "n_pay", "amt", "first_dt", "last_dt", "med_gap", "med_gap_bd", "med_amt", "n_gaps", "n_ontime",
                        "max_ids", F.col("_last.cat").alias("last_cat"), F.col("_last.idsrc").alias("last_idsrc"),
                        F.col("_last.cp_class").alias("cp_class"), F.col("_last.amt").alias("last_amt"),
                        F.concat(F.when(F.col("tm"), "TM ").otherwise("non-TM "),
                                 F.when(F.col("pt") == "O", "org").when(F.col("pt") == "P", "person").otherwise("unknown")).alias("segment")))
        st.write.mode("overwrite").parquet(P_ST)

def classify(st):
    cad = F.lit(None).cast("string")
    for name, lo, hi in CADENCES:
        cad = F.when(F.col("med_gap").between(lo, hi), F.lit(name)).otherwise(cad)
    cad = F.when(F.col("med_gap_bd") <= 1, F.lit("daily")).otherwise(cad)
    reg = (F.col("n_pay") >= MIN_PAYMENTS) & cad.isNotNull() & (F.col("n_ontime") / F.col("n_gaps") >= REG_MIN)
    w_end = F.to_date(F.lit(W_END.isoformat()))
    stopped = F.coalesce(reg & F.when(cad == "daily", (bidx(w_end) - bidx(F.col("last_dt"))) >= DAILY_STOP_BD)
                                .otherwise(F.datediff(w_end, F.col("last_dt")) >= float(SKIP_MULT) * F.col("med_gap")), F.lit(False))
    # monthly and longer cadences are paid on calendar dates: the next payment is due the same day 1 / 2 / 3 months later
    expected = (F.when(cad == "monthly", F.add_months("last_dt", 1)).when(cad == "bimonthly", F.add_months("last_dt", 2))
                 .when(cad == "quarterly", F.add_months("last_dt", 3))
                 .otherwise(F.expr("date_add(last_dt, cast(med_gap as int))")))
    ptype = (F.when(F.col("payer_kind") == "CUST", "PNC customer").when(F.col("cp_class") == "business", "business")
              .when(F.col("cp_class") == "person", "person").when(F.col("cp_class") == "bank", "bank").otherwise("other"))
    return (st.withColumn("cadence", cad).withColumn("regular", F.coalesce(reg, F.lit(False)))
              .withColumn("stopped", stopped).withColumn("expected_dt", F.when(stopped, expected))
              .withColumn("exp_month", F.date_format("expected_dt", "yyyy-MM"))
              .withColumn("payer_type", ptype)
              .withColumn("last_dom", F.dayofmonth("last_dt")))

ST = classify(spark.read.parquet(P_ST)).cache()
REG = ST.where("regular")
names = (spark.table(T_ENT).select(F.col("ent_id").alias("payer"), F.col("ent_name").alias("payer_name"))
              .unionByName(spark.table(T_CUST).select(F.col("mdm_id").alias("payer"), F.col("customer_name").alias("payer_name")))
              .groupBy("payer").agg(F.min("payer_name").alias("payer_name")))
s2 = (ST.groupBy("flow").agg(F.count("*").alias("streams"), F.sum(F.col("regular").cast("long")).alias("regular"),
                             F.sum(F.col("stopped").cast("long")).alias("stopped")).toPandas())
out(sec("§2 Incoming streams — " + (f"built ({TIMES['§2 streams']:.1f} min)" if st_built else "reused"), tbl(s2)),
    para(f"Regular: ≥ {MIN_PAYMENTS} payment days, known cadence, ≥ {REG_MIN:.0%} of gaps on time. Stopped: silent ≥ "
         f"{SKIP_MULT} × median gap at {W_END} (daily: {DAILY_STOP_BD} business days)."))

## §3 Test 1 — seasonal?

Stop rate by expected month and cadence over the whole history (stops / regular streams at risk in that month; a
stream is at risk from its first payment to its stop or the window end). The second table puts the same calendar month
of each year side by side for the focus cadence: a seasonal pattern repeats every year.

In [ ]:
# ---- §3 Seasonal ------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

w_end_c = F.to_date(F.lit(W_END.isoformat()))
risk = (REG.select("cadence", F.explode(F.sequence(F.trunc("first_dt", "month"),
                                                   F.trunc(F.coalesce("expected_dt", w_end_c), "month"),
                                                   F.expr("interval 1 month"))).alias("m"))
           .groupBy(F.date_format("m", "yyyy-MM").alias("month"), "cadence").count().withColumnRenamed("count", "at_risk")
           .toPandas())
stp = (REG.where("stopped").groupBy(F.col("exp_month").alias("month"), "cadence").count()
          .withColumnRenamed("count", "stops").toPandas())
mm = risk.merge(stp, on=["month", "cadence"], how="left").fillna({"stops": 0})
mm = mm[mm.month.isin(MONTHS)]
mt = mm.groupby("month")[["at_risk", "stops"]].sum().reset_index()
RATE = pd.DataFrame({"month": mt.month, "stops": mt.stops.astype("int64"), "at risk": mt.at_risk.astype("int64"),
                     "stop rate": mt.stops / mt.at_risk})
for cd in ["weekly", "biweekly / semimonthly", "monthly"]:
    x = mm[mm.cadence == cd].set_index("month")
    RATE[f"{cd} stops"] = RATE.month.map(x.stops).fillna(0).astype("int64")
    RATE[f"{cd} rate"] = RATE.month.map(x.stops / x.at_risk)
RATE = RATE.sort_values("month").reset_index(drop=True)
fr = RATE.set_index("month")[f"{FOCUS} rate"]
cand = [m for m in MONTHS[6:-2] if m in fr.index]
SPIKE = SPIKE_MONTH or (fr.loc[cand].idxmax() if cand else MONTHS[-3])
yoy = pd.DataFrame({"calendar month": [f"{i:02d}" for i in range(1, 13)]})
for y in sorted({m[:4] for m in MONTHS}):
    yoy[y] = yoy["calendar month"].map(lambda c: fr.get(f"{y}-{c}"))
same = [f"{int(SPIKE[:4]) - k}-{SPIKE[5:]}" for k in (1, 2)]
usual = fr.drop([SPIKE], errors="ignore").loc[[m for m in cand if m != SPIKE]].median()
SEAS = {"spike rate": fr.get(SPIKE), "usual rate (median, other months)": usual,
        "same month 1 year earlier": fr.get(same[0]), "same month 2 years earlier": fr.get(same[1])}
out(sec(f"§3a Stop rate by expected month (focus: {FOCUS}; spike month {SPIKE})",
        tbl(RATE, pct=tuple(c for c in RATE.columns if c.endswith("rate")))),
    sec(f"§3b {FOCUS} stop rate by calendar month and year", tbl(yoy, pct=tuple(yoy.columns[1:]))),
    sec("§3c Spike month against the usual month and the same month in earlier years",
        tbl(pd.DataFrame([{"measure": k, "stop rate": v} for k, v in SEAS.items()]), pct=("stop rate",))),
    para("The first months are low because a stream needs MIN_PAYMENTS payments before it can stop; the last two months "
         "are low because long cadences need time to look stopped."))

## §4 Test 2 — one source?

For stops of the focus cadence: how the spike month's stops split by the category of the last payment, payer type, id
source of the payer, flow and the receiving customer's segment, against the usual month (median of the other months).
**Excess** = spike-month stops minus usual stops; a data change shows as most of the excess in one row. Then, straight
from `pkg_edge_monthly` (all cadences, same sampled customers): the share of incoming payer–customer pairs of each
category that were active last month and are gone this month.

In [ ]:
# ---- §4 One source -----------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

FS = REG.where(F.col("stopped") & (F.col("cadence") == FOCUS))
other_months = [m for m in cand if m != SPIKE]
def excess(dim):
    g = FS.groupBy("exp_month", dim).count().toPandas()
    p = g.pivot_table(index=dim, columns="exp_month", values="count", aggfunc="sum", fill_value=0)
    p = p.reindex(columns=sorted(set(p.columns) | {SPIKE}), fill_value=0)
    om = [m for m in other_months if m in p.columns]
    res = pd.DataFrame({dim: p.index.astype(str), "stops in spike month": p[SPIKE].astype("int64").values,
                        "usual month": (p[om].median(axis=1) if om else 0).values})
    res["excess"] = res["stops in spike month"] - res["usual month"]
    tot_ex = res.excess.clip(lower=0).sum()
    res["share of spike"] = res["stops in spike month"] / max(res["stops in spike month"].sum(), 1)
    res["share of excess"] = res.excess.clip(lower=0) / tot_ex if tot_ex > 0 else np.nan
    return res.sort_values("excess", ascending=False).head(12)
EX = {d: excess(d) for d in ["last_cat", "payer_type", "last_idsrc", "flow", "segment"]}
TOP_EXCESS = {d: float(x["share of excess"].max()) if len(x) else np.nan for d, x in EX.items()}

# edge level: incoming pairs lost per month and category (pkg_edge_monthly, sampled customers)
pairs_built = not exists(f"{P_PAIRS}/_SUCCESS")
if pairs_built:
    with timed("§4 edge pairs"):
        em = (spark.table(T_EDGE_M).where(F.col("month").isin(MONTHS))
                .where((F.col("dst_kind") == "CUST") & ~F.coalesce(F.col("is_self"), F.lit(False)) & sampled("dst_id"))
                .select("src_id", "dst_id", code_of.alias("cat"), "month").distinct()
                .withColumn("mi", F.expr("cast(substr(month, 1, 4) as int) * 12 + cast(substr(month, 6, 2) as int) - 1")))
        wp = Window.partitionBy("src_id", "dst_id", "cat").orderBy("mi")
        em = em.withColumn("next_mi", F.lead("mi").over(wp))
        last_mi = int(END[:4]) * 12 + int(END[5:7]) - 1
        lost = (em.where((F.col("mi") < last_mi) & (F.col("next_mi").isNull() | (F.col("next_mi") != F.col("mi") + 1)))
                  .groupBy((F.col("mi") + 1).alias("mi"), "cat").agg(F.count("*").alias("lost")))
        act = em.groupBy("mi", "cat").agg(F.count("*").alias("active"))
        (act.withColumn("mi", F.col("mi") + 1).withColumnRenamed("active", "active_prev")
            .join(lost, ["mi", "cat"], "left").fillna({"lost": 0})
            .write.mode("overwrite").parquet(P_PAIRS))
pl = spark.read.parquet(P_PAIRS).toPandas()
pl["month"] = pl.mi.map(lambda v: f"{v // 12:04d}-{v % 12 + 1:02d}")
pl = pl[pl.month.isin(MONTHS[1:])]
pl["lost share"] = pl.lost / pl.active_prev
lp = pl.pivot_table(index="cat", columns="month", values="lost share")
big = pl.groupby("cat").active_prev.median().sort_values(ascending=False)
lp = lp.reindex(index=[c for c in big.index if big[c] >= 100])
om2 = [m for m in other_months if m in lp.columns]
LOST = pd.DataFrame({"category": lp.index, "pairs active (median month)": big.reindex(lp.index).round().astype("int64").values,
                     "lost in spike month": lp[SPIKE].values if SPIKE in lp else np.nan,
                     "lost, usual month": lp[om2].median(axis=1).values if om2 else np.nan})
LOST["ratio"] = LOST["lost in spike month"] / LOST["lost, usual month"]
LOST = LOST.sort_values("ratio", ascending=False)
blocks = [sec(f"§4{chr(97 + i)} {FOCUS} stops in {SPIKE} by {d}", tbl(x, pct=("share of spike", "share of excess"), dec=("usual month", "excess")))
          for i, (d, x) in enumerate(EX.items())]
out(*blocks,
    sec(f"§4f Incoming payer–customer pairs gone from the edges, by category (pkg_edge_monthly): {SPIKE} against the usual month",
        tbl(LOST, pct=("lost in spike month", "lost, usual month"), dec=("ratio",))),
    para("last_cat = category of the stream's last payment (2C = ACH from a counterparty, 3C = ACH from a non-PNC "
         "originator, 5C / 6C = RTP, 7C = check, 4C = wire, 1A / 2A / 4A / 5A / 6A = book). last_idsrc: A = account id, "
         "B = name-hash id, internal = book."))

## §5 Test 3 — replaced?

For every stopped regular stream: does the same customer start receiving money from **another** payer between
`REPL_BEFORE` days before and `REPL_AFTER` days after the expected payment, with a usual amount within ±`REPL_AMT_TOL`?
If so the stream was most likely **replaced**: by the same payer under a new id or entity (same name), or by a
different payer (another name, e.g. a new payroll provider or a tenant paying from a new account). A spike of
replacements with the same name points to an id change in the data.

In [ ]:
# ---- §5 Replaced -------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

S = (REG.where("stopped").select("payee", F.col("payer").alias("old"), "cadence", "exp_month", "expected_dt",
                                 F.col("med_amt").alias("old_amt")))
N_ = ST.select("payee", F.col("payer").alias("new"), F.col("first_dt").alias("new_first"), F.col("med_amt").alias("new_amt"),
               F.col("cadence").alias("new_cadence"))
cand_ = (S.join(N_, "payee")
          .where((F.col("new") != F.col("old"))
                 & F.col("new_first").between(F.date_sub("expected_dt", REPL_BEFORE), F.date_add("expected_dt", REPL_AFTER))
                 & (F.abs(F.col("new_amt") - F.col("old_amt")) <= REPL_AMT_TOL * F.col("old_amt"))))
nmo = names.select(F.col("payer").alias("old"), F.col("payer_name").alias("old_name"))
nmn = names.select(F.col("payer").alias("new"), F.col("payer_name").alias("new_name"))
norm = lambda c: F.regexp_replace(F.upper(F.coalesce(c, F.lit(""))), "[^A-Z0-9]", "")
best = (cand_.join(nmo, "old", "left").join(nmn, "new", "left")
             .withColumn("same_name", (norm(F.col("old_name")) == norm(F.col("new_name"))) & (norm(F.col("old_name")) != ""))
             .withColumn("d", F.abs(F.col("new_amt") - F.col("old_amt")))
             .withColumn("r", F.row_number().over(Window.partitionBy("payee", "old").orderBy(F.desc("same_name"), "d", "new")))
             .where("r = 1"))
rep = S.join(best.select("payee", "old", "new", "same_name", "old_name", "new_name"), ["payee", "old"], "left")
rm_ = (rep.groupBy("exp_month", "cadence")
          .agg(F.count("*").alias("stops"), F.sum(F.col("new").isNotNull().cast("long")).alias("replaced"),
               F.sum(F.coalesce(F.col("same_name"), F.lit(False)).cast("long")).alias("replaced, same name"))
          .toPandas())
rf = rm_[rm_.cadence == FOCUS].copy()
rf = rf[rf.exp_month.isin(MONTHS)].sort_values("exp_month")
rf["replaced share"] = rf.replaced / rf.stops
rf["same-name share"] = rf["replaced, same name"] / rf.stops
REPL = {"spike": rf.set_index("exp_month")["replaced share"].get(SPIKE),
        "usual": rf[rf.exp_month.isin(other_months)]["replaced share"].median(),
        "spike same name": rf.set_index("exp_month")["same-name share"].get(SPIKE),
        "usual same name": rf[rf.exp_month.isin(other_months)]["same-name share"].median()}
pairs_top = (rep.where((F.col("exp_month") == SPIKE) & (F.col("cadence") == FOCUS) & F.col("new").isNotNull())
                .groupBy("old_name", "new_name", "same_name").count().orderBy(F.desc("count")).limit(15).toPandas())
out(sec(f"§5a {FOCUS} stops replaced by a new payer within the window, by expected month",
        tbl(rf.drop(columns=["cadence"]).rename(columns={"exp_month": "month"}), pct=("replaced share", "same-name share"))),
    sec(f"§5b Most common replacements in {SPIKE} (old payer → new payer)", tbl(pairs_top)),
    para(f"Spike month: {REPL['spike'] or 0:.1%} replaced ({REPL['spike same name'] or 0:.1%} by the same name); "
         f"usual month: {REPL['usual'] or 0:.1%} ({REPL['usual same name'] or 0:.1%})."))

## §6 Test 4 — real life?

For stops of the focus cadence, spike month against the usual month: whether the receiving customer was still active
after the expected payment (any payment in `pkg_edge_monthly` in a later month), the day of the month of the last
payment (rent and leases are paid on the 1st–5th), the amount, payer-wide stops, and the payers with the most stops in
the spike month next to the same month a year earlier.

In [ ]:
# ---- §6 Real life ------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

cl = (spark.table(T_EDGE_M).where(F.col("month").isin(MONTHS) & ~F.coalesce(F.col("is_self"), F.lit(False)))
           .select(F.when(F.col("dst_kind") == "CUST", F.col("dst_id")).alias("c1"),
                   F.when(F.col("src_kind") == "CUST", F.col("src_id")).alias("c2"), "month")
           .select(F.explode(F.array("c1", "c2")).alias("payee"), "month").where(F.col("payee").isNotNull() & sampled("payee"))
           .groupBy("payee").agg(F.max("month").alias("last_active")))
pwm = FS.groupBy("payer", "exp_month").agg(F.count("*").alias("_n"))
fx = (FS.join(cl, "payee", "left").join(pwm, ["payer", "exp_month"], "left")
        .withColumn("active_after", F.coalesce(F.col("last_active") > F.col("exp_month"), F.lit(False)))
        .withColumn("dom", F.when(F.col("last_dom") <= 5, "1st–5th").when(F.col("last_dom") <= 15, "6th–15th")
                           .when(F.col("last_dom") <= 25, "16th–25th").otherwise("26th–31st"))
        .withColumn("amount", F.when(F.col("med_amt") < 500, "< 500").when(F.col("med_amt") < 2000, "500–2k")
                              .when(F.col("med_amt") < 10000, "2k–10k").otherwise("≥ 10k"))
        .withColumn("payer_wide", F.col("_n") >= MASS_STOP_MIN)
        .withColumn("grp", F.when(F.col("exp_month") == SPIKE, "spike").when(F.col("exp_month").isin(other_months), "usual")))
fx = fx.where(F.col("grp").isNotNull()).cache()
n_usual_months = max(len(other_months), 1)
def split(dim):
    g = fx.groupBy("grp", dim).count().toPandas().pivot_table(index=dim, columns="grp", values="count", fill_value=0)
    for c in ("spike", "usual"):
        if c not in g:
            g[c] = 0
    res = pd.DataFrame({dim: g.index.astype(str), "spike month": g["spike"].astype("int64").values,
                        "usual month (avg)": (g["usual"] / n_usual_months).values})
    res["share, spike"] = res["spike month"] / max(res["spike month"].sum(), 1)
    res["share, usual"] = res["usual month (avg)"] / max(res["usual month (avg)"].sum(), 1e-9)
    return res
R6 = {d: split(d) for d in ["active_after", "dom", "amount", "payer_wide"]}
ACTIVE = R6["active_after"].set_index("active_after")
yr = f"{int(SPIKE[:4]) - 1}-{SPIKE[5:]}"
tp = (FS.where(F.col("exp_month").isin(SPIKE, yr)).groupBy("payer", "exp_month").count().toPandas()
        .pivot_table(index="payer", columns="exp_month", values="count", fill_value=0))
for c in (SPIKE, yr):
    if c not in tp:
        tp[c] = 0
tp = tp.sort_values(SPIKE, ascending=False).head(20).reset_index()
nm_ = names.where(F.col("payer").isin(list(tp.payer))).toPandas().set_index("payer").payer_name
tp.insert(0, "payer_name", tp.payer.map(nm_))
tp = tp.rename(columns={SPIKE: f"stops {SPIKE}", yr: f"stops {yr}"})
fx.unpersist()
out(*[sec(f"§6{chr(97 + i)} {FOCUS} stops by {d}: spike month against the usual month",
          tbl(x, pct=("share, spike", "share, usual"), dec=("usual month (avg)",))) for i, (d, x) in enumerate(R6.items())],
    sec(f"§6e Payers with the most {FOCUS} stops in {SPIKE}, and in {yr}", tbl(tp)),
    para("active_after: the receiving customer made or received any payment in a later month. payer_wide: the payer "
         f"stopped at least {MASS_STOP_MIN} sampled customers in the same month."))

## §7 Verdict

Each test gives one number; the reading follows fixed rules so the answer does not depend on who reads the tables.

In [ ]:
# ---- §7 Verdict --------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

def f(v):
    return None if v is None or (isinstance(v, float) and np.isnan(v)) else float(v)
sp, us, ly = f(SEAS["spike rate"]), f(SEAS["usual rate (median, other months)"]), f(SEAS["same month 1 year earlier"])
spike_x = sp / us if sp and us else None
seasonal = bool(ly and us and sp and (ly / us) >= 1.5 and (ly / sp) >= 0.6)
cat_top = EX["last_cat"].iloc[0] if len(EX["last_cat"]) else None
one_cat = bool(cat_top is not None and f(cat_top["share of excess"]) and f(cat_top["share of excess"]) >= 0.5)
lost_top = LOST.iloc[0] if len(LOST) else None
lost_jump = bool(lost_top is not None and f(lost_top["ratio"]) and f(lost_top["ratio"]) >= 2)
rs, ru = f(REPL["spike"]) or 0, f(REPL["usual"]) or 0
rss, rus = f(REPL["spike same name"]) or 0, f(REPL["usual same name"]) or 0
replaced = rs >= 0.3 and rs >= 2 * max(ru, 0.01)
same_name = rss >= 0.2 and rss >= 2 * max(rus, 0.01)
act_s = f(ACTIVE["share, spike"].get("True")) or 0
act_u = f(ACTIVE["share, usual"].get("True")) or 0
pw = R6["payer_wide"].set_index("payer_wide")
pw_s = f(pw["share, spike"].get("True")) or 0
rows = [
    {"test": "size of the spike", "value": f"{spike_x:.1f}× the usual {FOCUS} stop rate" if spike_x else "—",
     "reading": "a real spike" if spike_x and spike_x >= 1.5 else "no clear spike"},
    {"test": "§3 seasonal", "value": f"same month a year earlier: {ly:.2%} vs usual {us:.2%}" if ly and us else "no earlier year",
     "reading": "repeats every year → seasonal" if seasonal else "does not repeat a year earlier"},
    {"test": "§4 one category", "value": f"{cat_top['last_cat']}: {f(cat_top['share of excess']):.0%} of the excess" if cat_top is not None and f(cat_top['share of excess']) else "—",
     "reading": "concentrated in one category" if one_cat else "spread over categories"},
    {"test": "§4 pairs gone from the edges", "value": f"{lost_top['category']}: {f(lost_top['ratio']):.1f}× the usual share" if lost_top is not None and f(lost_top['ratio']) else "—",
     "reading": "a category lost pairs in the spike month" if lost_jump else "no category lost unusually many pairs"},
    {"test": "§5 replaced", "value": f"{rs:.0%} replaced ({rss:.0%} same name) vs usual {ru:.0%} ({rus:.0%})",
     "reading": ("replaced by the same payer under a new id → data / id change" if same_name else
                 "replaced by other payers → real switches" if replaced else "mostly not replaced")},
    {"test": "§6 customers still active", "value": f"{act_s:.0%} in the spike month vs {act_u:.0%} usual",
     "reading": "customers carry on: the payment ended, not the customer" if act_s >= 0.8 else "many customers went quiet too"},
    {"test": "§6 payer-wide", "value": f"{pw_s:.0%} of spike stops are payer-wide", "reading": "a few payers" if pw_s >= 0.5 else "many payers"}]
if seasonal:
    verdict = f"Seasonal: {SPIKE[5:]} has the same jump in earlier years. Treat it as a real calendar effect (lease and school-year endings), not a data problem."
elif same_name:
    verdict = "Data / id change: the stopped streams continue under new payer ids with the same names. Treat the spike as an artifact: the stops are not real."
elif one_cat and lost_jump:
    verdict = f"Data change in category {cat_top['last_cat']}: the excess sits there and its pairs vanished from the edges. Check that feed with the data team."
elif replaced:
    verdict = "Real switches: customers kept receiving the same amount from new payers (new payroll providers, accounts). Real events, but not losses of income."
else:
    verdict = "Real stops: not seasonal in earlier years, no single source, not replaced. The payments ended; the customers carried on."
out(sec(f"§7 Verdict on {SPIKE} ({FOCUS} streams)", tbl(pd.DataFrame(rows))), sec("Verdict", para(verdict)),
    para("Rules: seasonal = same month a year earlier ≥ 1.5× the usual rate and ≥ 60% of the spike; one category = "
         "≥ 50% of the excess; pairs gone = ≥ 2× the usual share; replaced = ≥ 30% and ≥ 2× usual; same name = ≥ 20% "
         "and ≥ 2× usual. The first rule that holds, in this order, gives the verdict."))

## §8 Figures

PNG files in `FIG_DIR/<window end>/`, nothing drawn inline.

In [ ]:
# ---- §8 Figures -------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
FDIR = os.path.join(FIG_DIR, END); os.makedirs(FDIR, exist_ok=True)
SURF, INK, INK2, GRID, NEUTRAL = "#fcfcfb", "#0b0b0b", "#52514e", "#e3e2de", "#c3c2b7"
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF, "axes.edgecolor": NEUTRAL,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
                     "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 9, "axes.titlesize": 10,
                     "axes.titleweight": "bold", "axes.titlelocation": "left", "text.color": INK, "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2, "legend.frameon": False, "lines.linewidth": 2})
SAVED = []
def save(fig, name, what):
    fig.savefig(os.path.join(FDIR, name), dpi=160, bbox_inches="tight"); plt.close(fig)
    SAVED.append({"file": name, "shows": what})
CAD_COLOR = {"weekly": PAL[1], "biweekly / semimonthly": PAL[2], "monthly": PAL[3]}

fig, ax = plt.subplots(figsize=(10, 3.4)); x = np.arange(len(RATE))
for cd in ["weekly", "biweekly / semimonthly", "monthly"]:
    ax.plot(x, RATE[f"{cd} rate"], color=CAD_COLOR[cd], marker="o", markersize=3, label=cd)
ax.axvline(list(RATE.month).index(SPIKE), color=NEUTRAL, linestyle="--", linewidth=1)
ax.set_xticks(x[::2]); ax.set_xticklabels(list(RATE.month)[::2], rotation=45); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(); ax.set_title("Stop rate of regular incoming streams by expected month")
save(fig, "01_stop_rate_history.png", "stop rate by month and cadence, whole history")

fig, ax = plt.subplots(figsize=(7, 3.2)); xs = np.arange(12)
for i, y in enumerate(yoy.columns[1:]):
    ax.plot(xs, yoy[y], color=PAL[i], marker="o", markersize=4, label=y)
ax.set_xticks(xs); ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.legend(); ax.set_title(f"{FOCUS.capitalize()} stop rate by calendar month, each year")
save(fig, "02_seasonality.png", "same calendar month across years")

x5 = rf.reset_index(drop=True)
fig, ax = plt.subplots(figsize=(10, 3.2)); xi = np.arange(len(x5))
ax.bar(xi, x5["replaced share"] - x5["same-name share"], color=PAL[0], label="replaced by another payer", edgecolor=SURF, linewidth=1)
ax.bar(xi, x5["same-name share"], bottom=x5["replaced share"] - x5["same-name share"], color=PAL[1], label="replaced, same name", edgecolor=SURF, linewidth=1)
ax.set_xticks(xi[::2]); ax.set_xticklabels(list(x5.exp_month)[::2], rotation=45); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=2); ax.grid(axis="x", visible=False)
ax.set_title(f"Share of {FOCUS} stops replaced by a new payer within weeks")
save(fig, "03_replaced.png", "replacement share by expected month")

lt = LOST.head(8)
fig, ax = plt.subplots(figsize=(7, 3.0)); yi = np.arange(len(lt))
ax.barh(yi - 0.2, lt["lost, usual month"], height=0.4, color=NEUTRAL, label="usual month")
ax.barh(yi + 0.2, lt["lost in spike month"], height=0.4, color=PAL[0], label=SPIKE)
ax.set_yticks(yi); ax.set_yticklabels(list(lt.category)); ax.invert_yaxis(); ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(); ax.grid(axis="y", visible=False); ax.set_title("Incoming pairs gone from the edges, by category")
save(fig, "04_pairs_lost_by_category.png", "share of pairs lost per category, spike vs usual")
out(sec(f"§8 Figures: {len(SAVED)} PNG files in {os.path.abspath(FDIR)}", tbl(pd.DataFrame(SAVED))))

## §9 Housekeeping

In [ ]:
# ---- §9 Housekeeping ----------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

ST.unpersist()
sizes = {"stream days": size_gb(P_SD), "streams": size_gb(P_ST), "edge pairs": size_gb(P_PAIRS)}
if not KEEP_SCRATCH:
    for p in (P_SD, P_ST, P_PAIRS):
        delete(p)
times = pd.DataFrame([{"step": k, "minutes": v} for k, v in TIMES.items()])
out(sec("§9 Scratch", tbl(pd.DataFrame([{"scratch": k, "GB": v} for k, v in sizes.items()]), dec=("GB",))),
    sec("Build time", tbl(times, dec=("minutes",))) if len(times) else "",
    para("Scratch kept for reruns." if KEEP_SCRATCH else "Scratch deleted."))